# Qwen2-VL-2B × MMAD zero-shot benchmark
Runs the exact same 140-question, seven-task manifest used by the Cosmos 3 UI bot. Outputs are checkpointed after every image.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.49,<5', 'accelerate>=1.2', 'qwen-vl-utils>=0.0.8', 'remotezip>=0.12', 'pillow', 'pandas', 'matplotlib', 'seaborn'], check=True)

In [ ]:
from pathlib import Path
import os, subprocess, sys
WORK = Path('/content') if Path('/content').exists() else Path('/kaggle/working')
REPO = WORK / 'mini-world-model'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/anhsown/mini-world-model', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
BENCH = REPO / 'research' / 'mmad_model_benchmark'
assert BENCH.exists(), 'Public repo is missing research/mmad_model_benchmark'
os.chdir(BENCH)
sys.path.insert(0, str(BENCH))
print('benchmark root:', BENCH)

In [ ]:
# Deterministically build the shared 140-question manifest and range-download only selected images.
subprocess.run([sys.executable, 'prepare_subset.py'], check=True)
import json
manifest = json.loads((BENCH/'data/subset_manifest.json').read_text(encoding='utf-8'))
print(manifest['manifest_sha256'], len(manifest['records']))

In [ ]:
import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from qwen_vl_utils import process_vision_info
MODEL_ID = 'Qwen/Qwen2-VL-2B-Instruct'
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*28*28, max_pixels=768*28*28)
model = Qwen2VLForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto', attn_implementation='sdpa').eval()
print('device:', next(model.parameters()).device, 'VRAM GiB:', round(torch.cuda.memory_allocated()/2**30, 2))

In [ ]:
# Set LIMIT=0 for all 140 records; use a small value for a smoke test.
LIMIT = 0
import time
from datetime import datetime, timezone
from common.mmad import SYSTEM_PROMPT, append_jsonl, load_jsonl, parse_prediction
OUT = BENCH / 'outputs/qwen2_vl/predictions.jsonl'
done = {r['sample_id'] for r in load_jsonl(OUT) if r.get('status') in {'ok','parse_failure'}}
records = manifest['records'][:LIMIT or None]
for i, sample in enumerate(records, 1):
    if sample['sample_id'] in done:
        print(f'[{i}/{len(records)}] SKIP {sample["sample_id"]}')
        continue
    image = (BENCH/'data'/sample['image_file']).resolve()
    messages = [{'role':'system','content':SYSTEM_PROMPT}, {'role':'user','content':[{'type':'image','image':str(image)}, {'type':'text','text':sample['prompt']}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors='pt').to(model.device)
    started = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=8, do_sample=False)
    generated = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated)]
    raw = processor.batch_decode(generated, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()
    pred = parse_prediction(raw)
    row = {'sample_id':sample['sample_id'], 'model':MODEL_ID, 'backend':'Transformers FP16 SDPA', 'manifest_sha256':manifest['manifest_sha256'], 'status':'ok' if pred else 'parse_failure', 'prediction':pred, 'raw_response':raw, 'latency_seconds':round(time.perf_counter()-started,3), 'created_at':datetime.now(timezone.utc).isoformat()}
    append_jsonl(OUT, row)
    print(f'[{i}/{len(records)}] pred={pred} truth={sample["answer"]} latency={row["latency_seconds"]:.2f}s')
print('saved:', OUT)

In [ ]:
from common.mmad import evaluate_records, write_evaluation
summary, scored = evaluate_records(manifest, load_jsonl(OUT))
write_evaluation(OUT.parent, summary, scored)
print(json.dumps(summary, indent=2, ensure_ascii=False))

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
df = pd.DataFrame(scored)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
task = df[df.parse_valid].groupby('question_type').correct.mean().sort_values()
task.plot.barh(ax=axes[0], title='Accuracy by MMAD task', xlim=(0,1), color='#4c78a8')
source = df[df.parse_valid].groupby('source_dataset').correct.mean().sort_values()
source.plot.barh(ax=axes[1], title='Accuracy by source', xlim=(0,1), color='#59a14f')
lat = df.latency_seconds.dropna()
axes[2].hist(lat, bins=min(20, max(5, len(lat)//5)), color='#f28e2b'); axes[2].set_title('Inference latency'); axes[2].set_xlabel('seconds')
plt.tight_layout(); plt.savefig(OUT.parent/'benchmark_plots.png', dpi=160, bbox_inches='tight'); plt.show()

In [ ]:
import shutil
archive = shutil.make_archive(str(WORK/'qwen2_vl_mmad_results'), 'zip', OUT.parent)
print('DOWNLOAD:', archive)